# Notebook 5 — Data Preprocessing

**Goals**

1. Apply fixes to the issues identified in Notebook 4 — in the **right
   order**, because some steps depend on others.
2. Fill **time gaps** so every series has a regular index.
3. Forward/backward-fill *static* columns whose values were lost during
   reindexing.
4. Impute *dynamic* numeric columns with the right method for the data.
5. Treat *outliers* with winsorisation or replacement.
6. Verify with before/after plots.

>  **Order matters!** A common mistake is to impute first and then fill
> time gaps — that propagates wrong values into the inserted rows. The
> sequence below avoids this.

```text
   1. fill_time_gaps        ← regularise the time index
   2. ffill/bfill static    ← restore lost static-column values
   3. impute dynamic cols   ← fill remaining NaNs
   4. treat outliers        ← winsorise or replace
```


In [1]:
# ──────────────────────────────────────────────────────────────────────────
# COLAB SETUP — run this once at the top of every tutorial notebook.
# It installs plotly + statsmodels and makes the toolkit importable.
# If you are running locally (not in Colab) the !pip line is harmless.
# ──────────────────────────────────────────────────────────────────────────
!pip install -q plotly statsmodels
import sys, os
# If you uploaded forecasting_toolkit.zip to Colab, unzip it once:
#   !unzip -o forecasting_toolkit.zip
# Otherwise place forecasting_toolkit/ next to this notebook.
sys.path.insert(0, os.path.abspath('.'))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import plotly.io as pio
pio.renderers.default = 'colab'   # change to 'notebook' for local Jupyter


In [2]:
# ──────────────────────────────────────────────────────────────────────────
# POINT THIS AT YOUR DATASET — fill in the four lines below.
# Everything in this notebook works for ANY tabular sales/demand dataset
# (M5, Rossmann, Walmart Store Item Demand, custom CSVs, etc.).
#
#   DATA_PATH    – path to your CSV / Parquet file
#   DATE_COL     – name of the timestamp column
#   TARGET_COL   – name of the column you want to forecast
#   KEY_COLS     – list of columns that together identify ONE time series
#   STATIC_COLS  – columns constant within a key (e.g. store_type, category)
#   DYNAMIC_COLS – columns that vary in time within a key (e.g. promo, price)
#   FREQUENCY    – pandas offset alias: 'D','W','MS','H',…
# ──────────────────────────────────────────────────────────────────────────
DATA_PATH    = './datasets/rohlik_kaggle/sales_processed_tft.csv'
DATE_COL     = 'date'
TARGET_COL   = 'sales'
KEY_COLS     = ['unique_id']
STATIC_COLS  = ['warehouse','product_unique_id','name','L1_category_name_en','L2_category_name_en','L3_category_name_en','L4_category_name_en','country']
DYNAMIC_COLS = ['total_orders','sell_price_main','type_0_discount','type_1_discount','type_2_discount','type_3_discount','type_4_discount','type_5_discount','type_6_discount','holiday_name',
                'holiday','shops_closed','winter_school_holidays','school_holidays','weekday','week','month','day','is_month_start','is_month_end','quarter','weekend','days_since_2020']
FREQUENCY    = 'D'

from forecasting_toolkit import data_io
spec = data_io.make_spec(
    date_col=DATE_COL, target_col=TARGET_COL,
    key_cols=KEY_COLS, static_cols=STATIC_COLS,
    dynamic_cols=DYNAMIC_COLS, frequency=FREQUENCY,
)
df = data_io.load_data(DATA_PATH, spec)
print(f'Loaded {len(df):,} rows × {df.shape[1]} columns')
df.head()


Loaded 4,054,440 rows × 38 columns


,unique_id,date,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,month,year,prev_year,day,is_month_start,is_month_end,quarter,weekend,days_since_2020,country
0,0,2022-07-18,Budapest_1,5289.0,3.97,710.89,0.09,0.0,0.0,0.00000,...,7,2022,2022,18,False,False,3,0,929,Hungary
1,0,2022-07-19,Budapest_1,5255.0,73.36,710.89,1.00,0.0,0.0,0.00000,...,7,2022,2022,19,False,False,3,0,930,Hungary
2,0,2022-07-20,Budapest_1,5334.0,558.09,710.89,0.96,0.0,0.0,0.45045,...,7,2022,2022,20,False,False,3,0,931,Hungary
3,0,2022-07-21,Budapest_1,5459.0,14.03,710.89,0.06,0.0,0.0,0.45045,...,7,2022,2022,21,False,False,3,0,932,Hungary
4,0,2022-07-22,Budapest_1,5461.0,558.53,710.89,0.97,0.0,0.0,0.45045,...,7,2022,2022,22,False,False,3,0,933,Hungary


In [3]:
import forecasting_toolkit as ft
print(f'Starting rows: {len(df):,}')


Starting rows: 4,054,440


## 5.1 Fill time gaps

`fill_time_gaps` reindexes each forecast key onto a complete date range
at the spec's frequency. Rows that didn't exist are inserted with NaN on
all non-key columns and an `_was_imputed` indicator is set to `True`.

Two important options:

- `fill_value=0` — set inserted-row targets to 0. Use this when missing
  dates genuinely mean "no demand" (closed store, weekends).
- `fill_value=None` — keep targets as NaN. Use this when missing dates
  are *recording gaps* and the truth is unknown. We'll impute them in
  step 3.

We'll use `fill_value=None` here so the rest of the pipeline applies.
You can come back and change it depending on your domain.


In [4]:
df_gap = ft.preprocessing.fill_time_gaps(df, spec, fill_value=None,
                                         add_indicator=True)
print(f'Rows after gap fill : {len(df_gap):,}  (was {len(df):,})')
print(f'Newly inserted rows : {df_gap["_was_imputed"].sum():,}')
df_gap.head()


Rows after gap fill : 5,025,926  (was 4,054,440)
Newly inserted rows : 971,486


,date,unique_id,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,year,prev_year,day,is_month_start,is_month_end,quarter,weekend,days_since_2020,country,_was_imputed
0,2022-07-18,0,Budapest_1,5289.0,3.97,710.89,0.09,0.0,0.0,0.00000,...,2022.0,2022.0,18.0,False,False,3.0,0.0,929.0,Hungary,False
1,2022-07-19,0,Budapest_1,5255.0,73.36,710.89,1.00,0.0,0.0,0.00000,...,2022.0,2022.0,19.0,False,False,3.0,0.0,930.0,Hungary,False
2,2022-07-20,0,Budapest_1,5334.0,558.09,710.89,0.96,0.0,0.0,0.45045,...,2022.0,2022.0,20.0,False,False,3.0,0.0,931.0,Hungary,False
3,2022-07-21,0,Budapest_1,5459.0,14.03,710.89,0.06,0.0,0.0,0.45045,...,2022.0,2022.0,21.0,False,False,3.0,0.0,932.0,Hungary,False
4,2022-07-22,0,Budapest_1,5461.0,558.53,710.89,0.97,0.0,0.0,0.45045,...,2022.0,2022.0,22.0,False,False,3.0,0.0,933.0,Hungary,False


### Sanity-check: no time gaps remain

`detect_time_gaps` should now report 0 missing dates everywhere.


In [5]:
gaps_after = ft.quality.detect_time_gaps(df_gap, spec)
print(f'Max remaining missing_dates: {gaps_after["missing_dates"].max()}')


Max remaining missing_dates: 0


## 5.2 Forward/backward-fill static columns

Reindexing in step 1 inserted rows with NaN for *all* non-key columns —
including columns we declared as **static** (`STATIC_COLS`). Since those
columns don't change in time within a key, we can safely propagate the
known value:

- `ffill_static` — forward-fill within each key (typical case)
- `bfill_static` — backward-fill within each key (when only later rows
  are populated)
- `ffill_bfill_static` — do both, in order (safest default)


In [6]:
if STATIC_COLS:
    before = df_gap[STATIC_COLS].isna().sum().to_dict()
    df_static = ft.preprocessing.ffill_bfill_static(df_gap, spec)
    after = df_static[STATIC_COLS].isna().sum().to_dict()
    import pandas as pd
    display(pd.DataFrame({'before': before, 'after': after}))
else:
    df_static = df_gap.copy()
    print('No STATIC_COLS declared — skipping.')


,before,after
warehouse,971486,0
product_unique_id,971486,0
name,971486,0
L1_category_name_en,971486,0
L2_category_name_en,971486,0
L3_category_name_en,971486,0
L4_category_name_en,971486,0
country,971486,0


## 5.3 Impute the target and dynamic columns

The toolkit's `impute_missing` supports multiple strategies. Pick one
based on what the missingness *means* in your domain.

| Method            | When to prefer                                                           |
|-------------------|--------------------------------------------------------------------------|
| `mean` / `median` | Quick baseline. Median is safer on skewed sales data.                    |
| `mode`            | Categorical-like dynamic columns (e.g. promo flag).                      |
| `zero` / `constant` | When NaN really means "no event" (no promo, no sale).                  |
| `ffill` / `bfill` | Slow-moving dynamic columns (price, exchange rate).                      |
| `linear` / `time` | Smooth numeric series with short, isolated gaps.                         |
| `rolling_mean`    | Noisy but locally stationary series.                                     |
| `seasonal_naive_7`| Daily data with a clear weekly cycle. Fills `t` with `t-7`.              |

All methods run **per forecast key** by default (`group=True`).


### A side-by-side comparison

We'll grab one series, knock out a chunk of its observations, and run
several imputers to see how they differ visually.


In [7]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Pick one example series with a long history
example_key = df_static[KEY_COLS].drop_duplicates().iloc[0].to_dict()
single = ft.data_io.get_series(df_static, spec, example_key).sort_values(DATE_COL).copy()
y_clean = single[TARGET_COL].astype(float).reset_index(drop=True)
dates = single[DATE_COL].reset_index(drop=True)

# Knock out a contiguous block (~10% of the series) so the methods differ visibly
np.random.seed(0)
n = len(y_clean)
gap_start = n // 3
gap_len = max(5, n // 10)
gap_idx = list(range(gap_start, gap_start + gap_len))
y_with_nan = y_clean.copy()
y_with_nan.iloc[gap_idx] = np.nan

# Build a tiny helper dataframe so impute_missing can be applied per-method
def impute_series(method):
    tmp = pd.DataFrame({DATE_COL: dates, TARGET_COL: y_with_nan, **{k: example_key[k] for k in KEY_COLS}})
    out = ft.preprocessing.impute_missing(tmp, spec, cols=TARGET_COL, method=method)
    return out[TARGET_COL].values

methods = ['linear', 'ffill', 'rolling_mean', 'seasonal_naive_7', 'median', 'zero']
results = {m: impute_series(m) for m in methods}

# Plot a 2x3 grid
fig = make_subplots(rows=2, cols=3, subplot_titles=methods,
                    vertical_spacing=0.14, horizontal_spacing=0.06)
positions = [(1,1),(1,2),(1,3),(2,1),(2,2),(2,3)]
for (m, vals), (r, c) in zip(results.items(), positions):
    fig.add_trace(go.Scatter(x=dates, y=y_clean, mode='lines',
                             line=dict(color='#9CA3AF', width=1), name='original',
                             showlegend=(r==1 and c==1)), row=r, col=c)
    fig.add_trace(go.Scatter(x=dates, y=vals, mode='lines',
                             line=dict(color='#2563EB', width=1.6), name='imputed',
                             showlegend=(r==1 and c==1)), row=r, col=c)
fig.update_layout(height=560, template='plotly_white',
                  title='Imputation methods compared (gap injected for illustration)')
fig.show()


**What you should notice**

- `linear` glides smoothly across the gap.
- `ffill` produces a flat plateau — the last seen value held constant.
- `rolling_mean` smooths but lags; sharp turning points are dampened.
- `seasonal_naive_7` recovers weekly pattern beautifully *if* the data
  has one.
- `median` / `zero` are blunt instruments — useful as defensive defaults
  but they hide structure.

The right choice depends on your data. There is no universal best.


### Apply the chosen method to the whole dataframe


In [8]:
chosen_method = 'linear'   # change as needed
df_imputed = ft.preprocessing.impute_missing(df_static, spec,
                                             cols=TARGET_COL,
                                             method=chosen_method)
remaining = df_imputed[TARGET_COL].isna().sum()
print(f'Remaining NaNs in {TARGET_COL} after "{chosen_method}" imputation: {remaining}')


Remaining NaNs in sales after "linear" imputation: 0


### Imputing dynamic numeric columns

The same function works on any numeric column. Pass a list to impute
several at once.


In [9]:
import pandas as pd
numeric_dynamic = [c for c in DYNAMIC_COLS
                   if c in df_imputed.columns
                   and pd.api.types.is_numeric_dtype(df_imputed[c])]
if numeric_dynamic:
    df_imputed = ft.preprocessing.impute_missing(df_imputed, spec,
                                                 cols=numeric_dynamic,
                                                 method='ffill')
    print(f'Imputed dynamic columns with ffill: {numeric_dynamic}')
else:
    print('No numeric DYNAMIC_COLS to impute.')


Imputed dynamic columns with ffill: ['total_orders', 'sell_price_main', 'type_0_discount', 'type_1_discount', 'type_2_discount', 'type_3_discount', 'type_4_discount', 'type_5_discount', 'type_6_discount', 'holiday', 'shops_closed', 'winter_school_holidays', 'school_holidays', 'weekday', 'week', 'month', 'day', 'quarter', 'weekend', 'days_since_2020']


## 5.4 Outlier treatment

You have two main tools:

| Tool                           | What it does                                       | When to use                       |
|--------------------------------|----------------------------------------------------|-----------------------------------|
| `winsorize`                    | Caps values at chosen percentiles (e.g. 1% / 99%)  | You want to keep the row but limit influence |
| `replace_outliers_with`        | Detects + replaces with median / mean / NaN       | You want to neutralise specific anomalies |

Both run per-series (`group=True`) by default. Setting outliers to NaN
and re-imputing is a popular two-step combo: detect with IQR, replace
with NaN, then `impute_missing(method='linear')`.

>  **Don't blindly remove outliers.** A spike on Black Friday is
> *information*, not noise. Inspect a few flagged points before deciding.


### Strategy A — Winsorise at 1% / 99%


In [10]:
df_wins = ft.preprocessing.winsorize(df_imputed, spec,
                                     col=TARGET_COL,
                                     lower_pct=0.01, upper_pct=0.99,
                                     group=True)
delta = (df_imputed[TARGET_COL] - df_wins[TARGET_COL]).abs()
print(f'Rows changed by winsorisation : {(delta > 0).sum():,}')
print(f'Mean magnitude of change      : {delta.mean():.3f}')


Rows changed by winsorisation : 99,916
Mean magnitude of change      : 0.708


### Strategy B — Replace IQR outliers with NaN, then re-impute


In [11]:
# Step 1: replace IQR-flagged outliers with NaN
df_repl = ft.preprocessing.replace_outliers_with(
    df_imputed, spec, col=TARGET_COL,
    detector=ft.quality.detect_outliers_iqr,
    replace_with='nan', group=True)

n_replaced = df_repl[TARGET_COL].isna().sum() - df_imputed[TARGET_COL].isna().sum()
print(f'Outliers set to NaN: {n_replaced}')

# Step 2: re-impute the holes with linear interpolation
df_repl = ft.preprocessing.impute_missing(df_repl, spec, cols=TARGET_COL, method='linear')
print(f'Remaining NaNs after re-imputation: {df_repl[TARGET_COL].isna().sum()}')


Outliers set to NaN: 215195
Remaining NaNs after re-imputation: 0


### Visual comparison: original vs winsorised vs replaced

Showing one strongly-affected key:


In [12]:
import plotly.graph_objects as go

# Find a series with the most outliers in the original data so the difference is visible
out_summary = ft.quality.outlier_summary(df_imputed, spec, method='iqr')
worst = out_summary.iloc[0][KEY_COLS].to_dict()

orig = ft.data_io.get_series(df_imputed, spec, worst).sort_values(DATE_COL)
wins = ft.data_io.get_series(df_wins,    spec, worst).sort_values(DATE_COL)
repl = ft.data_io.get_series(df_repl,    spec, worst).sort_values(DATE_COL)

fig = go.Figure()
fig.add_trace(go.Scatter(x=orig[DATE_COL], y=orig[TARGET_COL], name='original',
                         line=dict(color='#9CA3AF', width=1)))
fig.add_trace(go.Scatter(x=wins[DATE_COL], y=wins[TARGET_COL], name='winsorised',
                         line=dict(color='#10B981', width=1.5)))
fig.add_trace(go.Scatter(x=repl[DATE_COL], y=repl[TARGET_COL], name='IQR→NaN→linear',
                         line=dict(color='#F59E0B', width=1.5, dash='dash')))
fig.update_layout(title=f'Outlier treatment comparison — {worst}',
                  template='plotly_white', height=440,
                  xaxis_title='Date', yaxis_title=TARGET_COL)
fig.show()


## 5.5 The cleaned dataframe

Pick the variant that suits your data and continue with that. For the
rest of the tutorial we'll use the *winsorised* version, which is the
least invasive choice.


In [13]:
df_clean = df_wins.copy()
print(f'Cleaned dataframe : {len(df_clean):,} rows × {df_clean.shape[1]} columns')
print(f'  • original rows           : {len(df):,}')
print(f'  • inserted by gap-filling : {df_clean["_was_imputed"].sum():,}')
print(f'  • NaNs in target          : {df_clean[TARGET_COL].isna().sum()}')
df_clean.head()


Cleaned dataframe : 5,025,926 rows × 39 columns
  • original rows           : 4,054,440
  • inserted by gap-filling : 971,486
  • NaNs in target          : 0


,date,unique_id,warehouse,total_orders,sales,sell_price_main,availability,type_0_discount,type_1_discount,type_2_discount,...,year,prev_year,day,is_month_start,is_month_end,quarter,weekend,days_since_2020,country,_was_imputed
0,2022-07-18,0,Budapest_1,5289.0,62.575579,710.89,0.09,0.0,0.0,0.00000,...,2022.0,2022.0,18.0,False,False,3.0,0.0,929.0,Hungary,False
1,2022-07-19,0,Budapest_1,5255.0,73.360000,710.89,1.00,0.0,0.0,0.00000,...,2022.0,2022.0,19.0,False,False,3.0,0.0,930.0,Hungary,False
2,2022-07-20,0,Budapest_1,5334.0,458.917500,710.89,0.96,0.0,0.0,0.45045,...,2022.0,2022.0,20.0,False,False,3.0,0.0,931.0,Hungary,False
3,2022-07-21,0,Budapest_1,5459.0,62.575579,710.89,0.06,0.0,0.0,0.45045,...,2022.0,2022.0,21.0,False,False,3.0,0.0,932.0,Hungary,False
4,2022-07-22,0,Budapest_1,5461.0,458.917500,710.89,0.97,0.0,0.0,0.45045,...,2022.0,2022.0,22.0,False,False,3.0,0.0,933.0,Hungary,False


## 5.6 Take-aways

- The pipeline is **gap-fill → static-fill → impute → outlier**. Skipping
  or reordering steps causes subtle bugs.
- Always keep `_was_imputed` available — many models benefit from knowing
  which observations are real.
- Imputation method choice is a *modelling decision*. Document it and
  re-evaluate later.
- Outliers are evaluated *per series*. A "high" value at one store can be
  a normal day at another.

Next: **Notebook 6 — Feature Engineering** — date features, lags,
rolling statistics, transformations, scaling and encoding.
